# ORNL Telemetry Data using Pandas, but vectorized

This is an attempt to transliterate the code in `Data_across_All_Caps.ipynb`
that uses pandas to work without any `for` loops, while trying to achieve
the same results.

This file is sort of a scratchpad. It does not do the full computation that
is in `Data_across_All_Caps.ipynb` and also does some other, extra
calculations that end up not being used.

There is not really any benefit in trying to make sense from this code.
From here we go to, `Data_across_all_caps_arkouda.ipynb`, which is also a
scratchpad, full of expriements.

For the final, vectorized arkouda implementation, see `Data_across_all_caps_arkouda_final.ipynb`

In [19]:
dir = "/lus/scratch/khandeka/hpegithub/ORNL-telemetry-analysis/parquet-traces-for-LSMS-application/"

power_cap200 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_200_16_events.parquet"
power_cap300 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_300_16_events.parquet"
power_cap400 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_400_16_events.parquet"
power_cap500 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_500_16_events.parquet"
power_cap600 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_600_16_events.parquet"
power_cap700 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_700_16_events.parquet"
power_cap800 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_800_16_events.parquet"
power_cap900 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_900_16_events.parquet"
power_cap1000 = "paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_1000_16_events.parquet"

In [25]:
import pandas as pd
import numpy as np
import time

df_events = pd.read_parquet(dir+power_cap500, columns=['time', 'metric', 'values', 'region','location','event'])


In [30]:
power_caps = [power_cap200, power_cap300, power_cap400, power_cap500, power_cap600, power_cap700, power_cap800, power_cap900, power_cap1000]
for power_cap in power_caps:
    df_events = pd.read_parquet(dir+power_cap, columns=['time', 'metric', 'values', 'region','location','event'])
    df_events["time"] = pd.to_numeric(df_events["time"])  # Convert time to seconds
    df_events["time"] = (df_events['time'] - df_events['time'].iloc[0]) / 1000000000
    df_gpupower = df_events[df_events["metric"] == "MetricInstance [21]: 'metric_class': MetricClass [20], 'recorder': Location [47244640256] '', 'metric_scope': MetricScope.SYSTEM_TREE_NODE, 'scope': SystemTreeNode [1] 'wombat35'"].copy()
    regions = df_events[df_events["region"].notna()]  # Filter out rows where region is NaN
    df_gpuevents = regions[regions["location"] == "CUDA[0:7]"].copy()
    df_gpuevents = df_gpuevents[df_gpuevents["region"]!= "nan"].copy()
    print(df_gpuevents)
    even = True
    last_time = 0
    for index, row in df_gpuevents.iterrows():
        if even and row['event'] != "Enter":
            print("Error: Enter event expected, got ", row['event'])
            print("At index: ", index)
            print(row)
            break
        if not even and row['event'] != "Leave":
            print("Error: Leave event expected, got ", row['event'])
            print("At index: ", index)
            print(row)
            break
        even = not even
        if last_time > row['time']:
            print("Error: Time decreasing")
            print("At index: ", index)
            print(row)
            print("Last time: ", last_time)
            break
        last_time = row['time']
    print("Done with ", power_cap)


                time metric values               region   location  event
33          0.052053    nan    nan         COMPUTE IDLE  CUDA[0:7]  Enter
73751       2.646692    nan    nan         COMPUTE IDLE  CUDA[0:7]  Leave
73752       2.646692    nan    nan  setDiagonalKerne...  CUDA[0:7]  Enter
73764       2.646718    nan    nan  setDiagonalKerne...  CUDA[0:7]  Leave
73765       2.646718    nan    nan         COMPUTE IDLE  CUDA[0:7]  Enter
...              ...    ...    ...                  ...        ...    ...
36282419  592.798531    nan    nan         COMPUTE IDLE  CUDA[0:7]  Leave
36282420  592.798531    nan    nan   copyTauToTau00Cuda  CUDA[0:7]  Enter
36282429  592.798570    nan    nan   copyTauToTau00Cuda  CUDA[0:7]  Leave
36282430  592.798570    nan    nan         COMPUTE IDLE  CUDA[0:7]  Enter
36285204  593.168324    nan    nan         COMPUTE IDLE  CUDA[0:7]  Leave

[2405378 rows x 6 columns]
Done with  paper_lsms_scorep_papi_wombat_grace_hopper_plugins_production_200_16_even

In [26]:
pd.set_option('display.max_columns', None)
pd.options.display.max_rows = 20
pd.set_option('display.max_colwidth', 20)
pd.set_option('display.width',1000)

print(df_events)

                     time               metric       values        region       location  event
0         259493900262272  MetricInstance [...     [982000]           nan                   nan
1         259493900262272  MetricInstance [...   [41904000]           nan                   nan
2         259493900262272  MetricInstance [...   [44849000]           nan                   nan
3         259493900262272  MetricInstance [...  [104743000]           nan                   nan
4         259493906325632  MetricInstance [...     [982000]           nan                   nan
...                   ...                  ...          ...           ...            ...    ...
35981836  259669405024864  MetricInstance [...  [209488000]           nan                   nan
35981837  259669410657120                  nan          nan  BUFFER FLUSH  Master thread  Enter
35981838  259669410672864                  nan          nan  BUFFER FLUSH  Master thread  Leave
35981839  259669410718336               

In [22]:
df_events["time"] = pd.to_numeric(df_events["time"])  # Convert time to seconds
print(df_events["time"])
df_events["time"] = (df_events['time'] - df_events['time'].iloc[0]) / 1000000000
print(df_events["time"])
df_gpupower = df_events[df_events["metric"] == "MetricInstance [21]: 'metric_class': MetricClass [20], 'recorder': Location [47244640256] '', 'metric_scope': MetricScope.SYSTEM_TREE_NODE, 'scope': SystemTreeNode [1] 'wombat35'"].copy()
print(df_gpupower)

0           260357149729472
1           260357149729472
2           260357149729472
3           260357149729472
4           260357155524736
                 ...       
35997905    260554632693856
35997906    260554632693856
35997907    260554632693856
35997908    260554632693856
35997909    260554645001504
Name: time, Length: 35997910, dtype: int64
0             0.000000
1             0.000000
2             0.000000
3             0.000000
4             0.005795
               ...    
35997905    197.482964
35997906    197.482964
35997907    197.482964
35997908    197.482964
35997909    197.495272
Name: time, Length: 35997910, dtype: float64
                time               metric    values region location event
1946        1.134165  MetricInstance [...   [77916]    nan            nan
1999        1.228771  MetricInstance [...   [82808]    nan            nan
2060        1.327646  MetricInstance [...   [86985]    nan            nan
2129        1.427583  MetricInstance [...   [91122]    

In [23]:
regions = df_events[df_events["region"].notna()]  # Filter out rows where region is NaN
df_gpuevents = regions[regions["location"] == "CUDA[0:7]"].copy()
df_gpuevents = df_gpuevents[df_gpuevents["region"]!= "nan"].copy()
print(df_gpuevents)

                time metric values               region   location  event
33          0.053534    nan    nan         COMPUTE IDLE  CUDA[0:7]  Enter
74043       3.094783    nan    nan         COMPUTE IDLE  CUDA[0:7]  Leave
74044       3.094783    nan    nan  setDiagonalKerne...  CUDA[0:7]  Enter
74056       3.094808    nan    nan  setDiagonalKerne...  CUDA[0:7]  Leave
74057       3.094808    nan    nan         COMPUTE IDLE  CUDA[0:7]  Enter
...              ...    ...    ...                  ...        ...    ...
35995184  197.192261    nan    nan         COMPUTE IDLE  CUDA[0:7]  Leave
35995185  197.192261    nan    nan   copyTauToTau00Cuda  CUDA[0:7]  Enter
35995194  197.192287    nan    nan   copyTauToTau00Cuda  CUDA[0:7]  Leave
35995195  197.192287    nan    nan         COMPUTE IDLE  CUDA[0:7]  Enter
35997904  197.482032    nan    nan         COMPUTE IDLE  CUDA[0:7]  Leave

[2405378 rows x 6 columns]


In [ ]:
# region_energy_sum, top_8_regions, region_runtimes = getInclusiveEnergy(df_gpuevents, df_gpupower)
# We break down this function into pieces

In [7]:
print(np.version.version)

1.26.4


In [8]:
df = df_gpuevents
df_power = df_gpupower
print(df)

df_power = df_power[df_power["values"] != "nan"]
df_power['values'] = df_power['values'].str.strip('[]').astype(float) / 1000


# start_time = time.time()

enter_events = df[df['event'] == 'Enter']
leave_events = df[df['event'] == 'Leave']
leave_events = leave_events.sort_values('time')
print(enter_events)

                time metric values  \
33          0.052053    nan    nan   
73751       2.646692    nan    nan   
73752       2.646692    nan    nan   
73764       2.646718    nan    nan   
73765       2.646718    nan    nan   
...              ...    ...    ...   
36282419  592.798531    nan    nan   
36282420  592.798531    nan    nan   
36282429  592.798570    nan    nan   
36282430  592.798570    nan    nan   
36285204  593.168324    nan    nan   

                                                region   location  event  
33                                        COMPUTE IDLE  CUDA[0:7]  Enter  
73751                                     COMPUTE IDLE  CUDA[0:7]  Leave  
73752     setDiagonalKernelCuda<std::complex<double> >  CUDA[0:7]  Enter  
73764     setDiagonalKernelCuda<std::complex<double> >  CUDA[0:7]  Leave  
73765                                     COMPUTE IDLE  CUDA[0:7]  Enter  
...                                                ...        ...    ...  
36282419          

                time metric values  \
33          0.052053    nan    nan   
73752       2.646692    nan    nan   
73765       2.646718    nan    nan   
73779       2.650696    nan    nan   
73801       2.658489    nan    nan   
...              ...    ...    ...   
36282406  592.798486    nan    nan   
36282408  592.798488    nan    nan   
36282418  592.798529    nan    nan   
36282420  592.798531    nan    nan   
36282430  592.798570    nan    nan   

                                                     region   location  event  
33                                             COMPUTE IDLE  CUDA[0:7]  Enter  
73752          setDiagonalKernelCuda<std::complex<double> >  CUDA[0:7]  Enter  
73765                                          COMPUTE IDLE  CUDA[0:7]  Enter  
73779                                    buildGijCudaKernel  CUDA[0:7]  Enter  
73801                                          COMPUTE IDLE  CUDA[0:7]  Enter  
...                                                     ...    

In [9]:
leave_times = leave_events['time'].values
leave_regions = leave_events['region'].values
print("Leave Times: ", leave_times)
print("Leave Regions: ", leave_regions)

enter_times = enter_events['time'].values
enter_regions = enter_events['region'].values
print("Enter Times: ", enter_times)
print("Enter Regions: ", enter_regions)

leave_indices = np.searchsorted(leave_times, enter_times, side='right')
print("Leave Indices: ", leave_indices)
valid_indices = (leave_indices < len(leave_times)) & (leave_regions[leave_indices] == enter_regions)
print("Valid Indices: ", valid_indices)

valid_enter_times = enter_times[valid_indices]
print("Valid Enter Times: ", valid_enter_times)
valid_leave_times = leave_times[leave_indices[valid_indices]]
print("Valid Leave Times: ", valid_leave_times)
valid_regions = enter_regions[valid_indices]
print("Valid Regions: ", valid_regions)

durations = valid_leave_times - valid_enter_times
print("Durations: ", durations)

region_times = pd.Series(durations, index=valid_regions).groupby(level=0).sum().to_dict()
sorted_region_times = dict(sorted(region_times.items()))
print(sorted_region_times)

Leave Times:  [  2.6466919    2.64671811   2.65069552 ... 592.79853088 592.79857004
 593.16832352]
Leave Regions:  ['COMPUTE IDLE' 'setDiagonalKernelCuda<std::complex<double> >'
 'COMPUTE IDLE' ... 'COMPUTE IDLE' 'copyTauToTau00Cuda' 'COMPUTE IDLE']
Enter Times:  [5.20531200e-02 2.64669190e+00 2.64671811e+00 ... 5.92798529e+02
 5.92798531e+02 5.92798570e+02]
Enter Regions:  ['COMPUTE IDLE' 'setDiagonalKernelCuda<std::complex<double> >'
 'COMPUTE IDLE' ... 'COMPUTE IDLE' 'copyTauToTau00Cuda' 'COMPUTE IDLE']
Leave Indices:  [      0       1       2 ... 1202686 1202687 1202688]
Valid Indices:  [ True  True  True ...  True  True  True]
Valid Enter Times:  [5.20531200e-02 2.64669190e+00 2.64671811e+00 ... 5.92798529e+02
 5.92798531e+02 5.92798570e+02]
Valid Leave Times:  [  2.6466919    2.64671811   2.65069552 ... 592.79853088 592.79857004
 593.16832352]
Valid Regions:  ['COMPUTE IDLE' 'setDiagonalKernelCuda<std::complex<double> >'
 'COMPUTE IDLE' ... 'COMPUTE IDLE' 'copyTauToTau00Cuda' 'CO

In [10]:
# Create a mask for the time intervals
energy_mask = (df_power['time'].values >= valid_enter_times[:, None]) & (df_power['time'].values <= valid_leave_times[:, None])
print("Energy Mask: ", energy_mask)
print(energy_mask.shape)

Energy Mask:  [[ True  True  True ... False False False]
 [False False False ... False False False]
 [False False False ... False False False]
 ...
 [False False False ... False False False]
 [False False False ... False False False]
 [False False False ...  True  True  True]]
(1202689, 5905)


In [12]:
# Apply the mask to get the relevant power and time values
energy_values = np.where(energy_mask, df_power['values'].values, 0)
print("Energy Values: ", energy_values)
print(energy_values.shape)

Energy Values:  [[ 77.518  81.842  85.916 ...   0.      0.      0.   ]
 [  0.      0.      0.    ...   0.      0.      0.   ]
 [  0.      0.      0.    ...   0.      0.      0.   ]
 ...
 [  0.      0.      0.    ...   0.      0.      0.   ]
 [  0.      0.      0.    ...   0.      0.      0.   ]
 [  0.      0.      0.    ... 144.075 141.336 138.29 ]]
(1202689, 5905)


In [13]:
energy_times = np.where(energy_mask, df_power['time'].values, 0)
print("Energy Times: ", energy_times)
print(energy_times.shape)

Energy Times:  [[  0.67687102   0.77262148   0.86653711 ...   0.           0.
    0.        ]
 [  0.           0.           0.         ...   0.           0.
    0.        ]
 [  0.           0.           0.         ...   0.           0.
    0.        ]
 ...
 [  0.           0.           0.         ...   0.           0.
    0.        ]
 [  0.           0.           0.         ...   0.           0.
    0.        ]
 [  0.           0.           0.         ... 592.89887383 593.01534433
  593.11812638]]
(1202689, 5905)


In [26]:
import scipy.integrate
# Calculate the energy using the trapezoidal rule
integrated_energies = []
# Takes 5 mins to run
for i in range(energy_mask.shape[0]):
  energy_value = df_power[energy_mask[i]]['values']
  energy_time = df_power[energy_mask[i]]['time']
  calc_energy = scipy.integrate.trapz(energy_value, energy_time)
  integrated_energies.append(calc_energy)
integrated_energies = np.array(integrated_energies)
print("Integrated Energies: ", integrated_energies)

/tmp/ipykernel_813141/2415936172.py:8: DeprecationWarning: 'scipy.integrate.trapz' is deprecated in favour of 'scipy.integrate.trapezoid' and will be removed in SciPy 1.14.0
  calc_energy = scipy.integrate.trapz(energy_value, energy_time)


Integrated Energies:  [203.48999671   0.           0.         ...   0.           0.
  45.0829951 ]


In [ ]:
# Sanity Check, not really part of the computation
print(valid_enter_times.shape)
print(valid_leave_times.shape)
print(energy_mask.shape)
for i, (enter_time, leave_time) in enumerate(zip(valid_enter_times, valid_leave_times)):
  power_df = df_power[(df_power['time'] >= enter_time) & (df_power['time'] <= leave_time)]
  mask_power_df = df_power[energy_mask[i]]
  if not power_df.equals(mask_power_df):
    print("Not equal at index: ", i)
    print("power_df :\n", power_df)
    print("mask_power_df :\n", mask_power_df)
    break

(1202689,)
(1202689,)
(1202689, 5905)


In [11]:
# Vectorized energy calculation
power_data_indices = np.searchsorted(df_power['time'].values, valid_enter_times)
print("Power Data Indices: ", power_data_indices, power_data_indices.shape)
last_tracked_powers = df_power['values'].iloc[power_data_indices - 1].values
print("Last Tracked Powers: ", last_tracked_powers, last_tracked_powers.shape)
print(df_power['values'].iloc[5904])
last_tracked_energies = last_tracked_powers * durations
print("Last Tracked Energies: ", last_tracked_energies, last_tracked_energies.shape)

Power Data Indices:  [   0   20   20 ... 5901 5901 5901] (1202689,)
Last Tracked Powers:  [138.29  118.906 118.906 ... 151.129 151.129 151.129] (1202689,)
138.29
Last Tracked Energies:  [3.58812597e+02 3.11616954e-03 4.72937913e-01 ... 2.22613017e-04
 5.91926955e-03 5.58804731e+01] (1202689,)


In [12]:
# Calculate energy for each region

energies = []
for mask, last_tracked_energy in zip(energy_mask, last_tracked_energies):
    power_data = df_power[mask]
    energy = np.trapz(power_data['values'], power_data['time']) if not power_data.empty else last_tracked_energy
    energies.append(energy)

In [ ]:
# Alternate way to do the same loop as the previous cell
other_energies = []
for i in range(energy_mask.shape[0]):
    power_data = df_power[energy_mask[i]]
    energy = np.trapz(power_data['values'], power_data['time']) if not power_data.empty else last_tracked_energies[i]
    other_energies.append(energy)

In [33]:
np.equal(energies, other_energies).all()

True

In [ ]:
# This approach did not help in vectorization, ignore this
power_data_empty = []
for mask in energy_mask:
    val = df_power[mask].empty
    power_data_empty.append(val)
power_data_empty = np.array(power_data_empty)
print("Power Data Empty: ", power_data_empty)

Power Data Empty:  [False  True  True ...  True  True False]


In [ ]:
# Calculate energy for each region
# Original, slow way from the pandas approach
energies = []
count = 0
for enter_time, leave_time in zip(valid_enter_times, valid_leave_times):
    power_data = df_power[(df_power['time'] >= enter_time) & (df_power['time'] <= leave_time)]
    if count < 3 or count > 1202686:
        print("-------------------------------------------------------------")
        print("----------------------- Iteration ", count, " ----------------------")
        print("Power Data Values: ", power_data['values'])
        print("Power Data Time: ", power_data['time'])
    if not power_data.empty:
        energy = np.trapz(power_data['values'], power_data['time'])
    else:
        energy = -1
    count += 1
    energies.append(energy)

In [14]:
y = np.array([4, 4, 4, 4, 4, 4, 4, 4])
x = np.array([0, 3, 3, 3, 3, 3, 3, 3])
print(np.trapz(y, x))

12.0


In [ ]:
# integrated_energeis = np.array(integrated_energies)
energies = np.array(energies)
print("Energies: ", energies)
print(energies.shape)
# final_energies = np.where(energies < 0, last_tracked_energies, energies)
final_energies = energies
print("Final Energies: ", final_energies)
print(final_energies.shape)
# Create a DataFrame from the energy data

energy_data = {
    'time': valid_enter_times,
    'event': ['Enter'] * len(valid_enter_times),
    'region': valid_regions,
    'energy': final_energies
}

df_energies = pd.DataFrame(energy_data)
region_energy_groupby = df_energies.groupby('region')['energy']
region_energy_sum = region_energy_groupby.sum().sort_values(ascending=False)
region_energy_sum = region_energy_sum[region_energy_sum > 1.0]
print("Region Energy Sum: \n", region_energy_sum)

# end_time = time.time()
# runtime = end_time - start_time
# print(f"Runtime: {runtime:.2f} seconds")

# Output the regions and their total energies
print("\n=== Total Energy per Region ===")
for region, energy in region_energy_sum.items():
    total_time = region_times[region]
    print(f"Region: {region}, Total Energy: {energy:.2f} J, Total Time: {total_time:.2f} s")

top_8_regions = region_energy_sum.head(8).index.tolist()
top_8_regions = [df[df['region'] == region].copy() for region in top_8_regions]

print(region_energy_sum.head(8))
print(top_8_regions)
print(region_times)



Energies:  [2.03489997e+02 3.11616954e-03 4.72937913e-01 ... 2.22613017e-04
 5.91926955e-03 4.50829951e+01]
(1202689,)
Final Energies:  [2.03489997e+02 3.11616954e-03 4.72937913e-01 ... 2.22613017e-04
 5.91926955e-03 4.50829951e+01]
(1202689,)
Region Energy Groupby: 
Region Energy Sum: 
 region
sm90_xmma_gemm_cf64cf64_f64f64_cf64_nn_n_tilesize64x64x32_stage3_warpsize4x2x1_tensor16x8x16_execute_kernel__5x_cublas            22473.631040
buildKKRMatrixMultiplyKernelCuda                                                                                                   8846.098688
sm90_xmma_gemm_cf64cf64_f64f64_cf64_nn_n_tilesize32x32x32_stage3_warpsize2x2x1_tensor16x8x16_execute_kernel__5x_cublas             4516.183488
getrf_pivot<getrf_params_<double2, 512, 2, 512, 64, 16> >                                                                          2845.945324
getrf_pivot<getrf_params_<double2, 512, 2, 512, 32, 32> >                                                                          2

In [3]:
import pandas as pd

# Example GPU DataFrame (gpudf)
gpudf = pd.DataFrame({
    'time': [1.5, 2.5, 3.5, 4.5],
    'other_metric': [10, 20, 30, 40]
})

# Example Power DataFrame (powerdf)
powerdf = pd.DataFrame({
    'time': [1.0, 2.0, 3.0, 4.0],
    'power': [100, 200, 300, 400]
})

# Ensure both DataFrames are sorted by 'time'
gpudf = gpudf.sort_values('time')
powerdf = powerdf.sort_values('time')

# Use merge_asof to align power values with GPU data
result = pd.merge_asof(gpudf, powerdf, on='time', direction='backward')

print(result)

   time  other_metric  power
0   1.5            10    100
1   2.5            20    200
2   3.5            30    300
3   4.5            40    400


In [ ]:
import numpy as np

# Example GPU DataFrame (gpudf)
gpudf_time = np.array([1.5, 2.5, 3.5, 4.5])
gpudf_other_metric = np.array([10, 20, 30, 40])

# Example Power DataFrame (powerdf)
powerdf_time = np.array([1.0, 2.0, 3.0, 4.0])
powerdf_power = np.array([100, 200, 300, 400])

# Ensure both arrays are sorted by time (Arkouda arrays are assumed sorted here)
# If not sorted, you can use ak.argsort to sort them.

# Find indices in powerdf_time where gpudf_time should align
indices = np.searchsorted(powerdf_time, gpudf_time, side='right') - 1

print(indices)

# Ensure indices are valid (i.e., not negative)
indices = np.where(indices < 0, 0, indices)

# Retrieve the corresponding power values
aligned_power = powerdf_power[indices]


# Combine the results into an Arkouda DataFrame
result = pd.DataFrame({
    'time': gpudf_time,
    'other_metric': gpudf_other_metric,
    'power': aligned_power
})

# Print the result
print(result)

[0 1 2 3]
   time  other_metric  power
0   1.5            10    100
1   2.5            20    200
2   3.5            30    300
3   4.5            40    400


In [ ]:
for enter, leave in zip(enter_events, leave_events):
  mask |= (time >= enter) & (time <= leave)

In [1]:
import numpy as np

# Time array
time = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

# Enter and leave events
enter_events = np.array([1, 6, 9])
leave_events = np.array([4, 7, 10])

# Use broadcasting to create a 2D mask for each enter-leave pair
# This is the problem operation that takes too much memory
# And the scaling is also dog water
mask = (time[:, None] >= enter_events) & (time[:, None] <= leave_events)
print(mask)
# Combine the masks using a logical "or" along the second axis
final_mask = mask.any(axis=1)

print(final_mask)

[[ True False False]
 [ True False False]
 [ True False False]
 [ True False False]
 [False False False]
 [False  True False]
 [False  True False]
 [False False False]
 [False False  True]
 [False False  True]]
[ True  True  True  True False  True  True False  True  True]


In [1]:
import numpy as np

# Time array
time = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

# Enter and leave events
enter_events = np.array([1, 6, 9])
leave_events = np.array([4, 7, 10])

# Create a mask initialized to False
mask = np.zeros_like(time, dtype=bool)

# Since arrays are sorted we can do this instead
# to efficiently compute the mask
enter_idx = np.searchsorted(time, enter_events, side='left')
leave_idx = np.searchsorted(time, leave_events, side='right')

# Mark ranges as True
# UGGHH its a for loop again
for start, end in zip(enter_idx, leave_idx):
    mask[start:end] = True

print(mask)

[ True  True  True  True False  True  True False  True  True]


In [2]:
import numpy as np

# Time array
time = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

# Enter and leave events
enter_events = np.array([1, 6, 9])
leave_events = np.array([4, 7, 10])

# Use sorted property to efficiently compute the mask
enter_idx = np.searchsorted(time, enter_events, side='left')
leave_idx = np.searchsorted(time, leave_events, side='right')
print(enter_idx)
print(leave_idx)

# Create a delta array to track changes
delta = np.zeros(len(time) + 1, dtype=int)
delta[enter_idx] += 1
delta[leave_idx] -= 1

# Compute the cumulative sum to get the final mask
mask = np.cumsum(delta[:-1]) > 0

print(mask)

[0 5 8]
[ 4  7 10]
[ True  True  True  True False  True  True False  True  True]


Input:

- time = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
- enter_events = [1, 6, 9]
- leave_events = [4, 7, 10]

Step 1: Delta Array
We use np.searchsorted to map enter_events and leave_events to indices in time:

enter_idx = [0, 5, 8] (indices of 1, 6, and 9 in time)
leave_idx = [4, 7, 10] (indices just after 4, 7, and 10 in time)
Now, we create a delta array:

Start with delta = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] (one extra element for boundary handling).
Add +1 at enter_idx: delta = [1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0].
Add -1 at leave_idx: delta = [1, 0, 0, 0, -1, 1, 0, -1, 1, 0, -1].
Step2:

Cumsum
[1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0]

slice[:-1]

[1, 1, 1, 1, 0, 1, 1, 0, 1, 1]

[ True  True  True  True False  True  True False  True  True]